In [36]:
import pandas as pd
import wandb
import optuna

from src.utils.config import reviews
from src.utils.files import read_file
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score,f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

from tqdm import tqdm

from utils_modelo_reviews.preprocesamiento import clean_text, train_val_test_split

In [40]:
def _preprocess(df):
    stop_words = set(stopwords.words("english"))
    ps = PorterStemmer()

    y = df["is_positive"]
    X = df["text"].progress_apply(lambda x : clean_text(x,ps,stop_words))
    
    return X, y


def build_objective(X_train, y_train, X_val, y_val):
    def objective(trial):
        tfidf = TfidfVectorizer(
            analyzer="word",
            ngram_range=(1,2),
            min_df=trial.suggest_int("min_df", 1, 5),
            max_df=trial.suggest_float("max_df", 0.7, 1.0),
            max_features=trial.suggest_categorical("max_features", [20000, 40000, 60000, None]),
            sublinear_tf=trial.suggest_categorical("sublinear_tf", [True, False]),
            strip_accents="unicode",
            lowercase=True,
        )

        clf = LogisticRegression(
            solver="saga",
            C=trial.suggest_float("C", 1e-2, 20.0, log=True),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
            max_iter=3000,
            random_state=42
        )

        pipe = Pipeline([
            ("tfidf", tfidf),
            ("clf", clf),
        ])
        
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_val)
        
        score = balanced_accuracy_score(y_val, y_pred)
        
        return score

    return objective

In [28]:
tqdm.pandas(desc="Limpiando texto")
print("Leyendo Datos")
df = read_file(reviews)
    
print("Preprocesado de los datos")
X, y = _preprocess(df)

Leyendo Datos
Preprocesado de los datos


Limpiando texto: 100%|██████████| 205883/205883 [03:19<00:00, 1033.89it/s]


In [38]:
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y)

In [42]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42)
    )

study.optimize(build_objective(X_train, y_train, X_val, y_val), n_trials=20, show_progress_bar= True)

Best trial: 16. Best value: 0.868063: 100%|██████████| 20/20 [09:43<00:00, 29.19s/it]


In [45]:
print("Mejores parámetros:", study.best_params)
print("Mejor balanced_accuracy:", study.best_value)

Mejores parámetros: {'min_df': 3, 'max_df': 0.8630302869833203, 'max_features': 40000, 'sublinear_tf': False, 'C': 1.3499190966166643, 'class_weight': 'balanced'}
Mejor balanced_accuracy: 0.8680630391449923
